# Totals Feature Engineering

Single source of truth for the 14 totals-specific features used by the totals model.
Loaded by `totals_model.ipynb` and `predict_totals.ipynb` via json+exec (same pattern as `features.ipynb`).

Public surface: `build_totals_features`, `TOTALS_FEATURE_COLS` (14 names), `totals_acc`.

All features are computed on top of the spread model's `g` DataFrame (which already contains
the 35 spread features). Keep this notebook separate from `features.ipynb` — totals features
must never leak into the spread pipeline.

## Parameters

In [ ]:
# Default ON for standalone runs. Consumer notebooks set False before loading.
RUN_TESTS = globals().get('RUN_TESTS', True)


## Imports

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

try:
    import nflreadpy as nfl
except ImportError as _e:
    raise ImportError(f'nflreadpy not found: {_e}') from _e

# As-of lookups + the fail-closed live preflight. `totals_asof.py` is plain importable
# Python (tested by betting/test_totals_asof.py and betting/test_totals_live.py) — the
# rolling features below are resolved by (team, season, week), never by the target game's
# game_id existing in a history table.
_TA_DIR = next((p for p in [Path.cwd() / 'betting', Path.cwd(), Path.cwd().parent / 'betting']
                if (p / 'totals_asof.py').exists()), None)
assert _TA_DIR is not None, 'totals_asof.py not found; tried betting/, CWD and ../betting'
if str(_TA_DIR) not in sys.path:
    sys.path.insert(0, str(_TA_DIR))
from totals_asof import (  # noqa: E402
    team_asof_rolling, league_asof_rolling,
    totals_live_preflight, TotalsPreflightError,
    TEMP_FALLBACK_F, WIND_FALLBACK_MPH, DOME_TEMP_F, DOME_WIND_MPH,
    TOTALS_FEATURE_COLS_EXPECTED,
)

if RUN_TESTS:
    print('Imports OK.')


## Constants

`TOTALS_FEATURE_COLS` — the 14 totals-specific feature columns in canonical order.
**DO NOT reorder** — column order determines `X_tr` shape which determines pkl identity.

In [ ]:
TOTALS_FEATURE_COLS = [
    # Vegas inputs (3)
    'total_line',
    'home_implied_pts',
    'away_implied_pts',
    # Weather + dome (3)
    'temp_f',
    'wind_mph',
    'is_dome',
    # Rolling scoring (5)
    'home_pts_scored_5g',
    'home_pts_allowed_5g',
    'away_pts_scored_5g',
    'away_pts_allowed_5g',
    'combined_pts_5g',
    # League environment (1)
    'league_avg_total_4wk',
    # Pace + matchup type (2)
    'pace_5g',
    'div_game',
]

# Cross-pin: the preflight in totals_asof.py keeps its own literal copy of this list so a
# caller cannot narrow what gets checked. The two must never drift apart.
assert TOTALS_FEATURE_COLS == TOTALS_FEATURE_COLS_EXPECTED, (
    'TOTALS_FEATURE_COLS has drifted from totals_asof.TOTALS_FEATURE_COLS_EXPECTED:\n'
    f'  notebook: {TOTALS_FEATURE_COLS}\n  totals_asof: {TOTALS_FEATURE_COLS_EXPECTED}')

if RUN_TESTS:
    assert len(TOTALS_FEATURE_COLS) == 14, f'Expected 14 totals features, got {len(TOTALS_FEATURE_COLS)}'
    assert len(TOTALS_FEATURE_COLS) == len(set(TOTALS_FEATURE_COLS)), 'Duplicate column names'
    print(f'Constants OK: {len(TOTALS_FEATURE_COLS)} totals feature cols')


## Helper — `totals_acc`

Hit-rate against the Vegas total. PUSH games (actual == line) are excluded from
both numerator and denominator (refund convention).

In [ ]:
def totals_acc(preds, total_line, actual_total):
    pred_over   = preds > total_line
    actual_over = actual_total > total_line
    push        = actual_total == total_line
    valid       = ~push
    if not valid.any():
        return 0.0
    return float(((pred_over == actual_over) & valid).sum()) / float(valid.sum())

if RUN_TESTS:
    import numpy as _np
    _p  = _np.array([45.0, 48.0, 41.0, 50.0])
    _tl = _np.array([44.5, 48.0, 43.5, 48.5])
    _a  = _np.array([47.0, 48.0, 40.0, 51.0])
    # game 1: pred OVER, actual OVER -> correct
    # game 2: push -> excluded
    # game 3: pred UNDER, actual UNDER -> correct
    # game 4: pred OVER, actual OVER -> correct
    assert abs(totals_acc(_p, _tl, _a) - 1.0) < 1e-6, 'totals_acc should be 1.0'
    print('totals_acc OK')


## Feature Group — `build_totals_features`

**Inputs:** `g` (spread model games DataFrame with 35 spread features already present),
`sched` (nflreadpy schedule DataFrame — **pass the COMPLETE schedule including the target
week**, not a completed-games-only slice; the scoring history is derived from it internally),
`pbp_full` (raw PBP DataFrame), `weather_path` (Path to `nfl_weather_*.csv` or None),
`impute_missing` (True = legacy training behaviour, False = live fail-closed behaviour).

**Outputs:** `g` with the 14 columns in `TOTALS_FEATURE_COLS`, plus the two provenance
columns `temp_f_source` / `wind_mph_source`.

**Note on `is_dome`:** `g['roof']` is ordinal-encoded by the spread pipeline (mc cell 33).
We re-merge the raw roof string from `sched` to correctly detect dome/closed-roof games.
Without this fix, `is_dome` is always 0.

**As-of rolling (2026-08-03).** Team scoring, pace and the league environment come from
`totals_asof.team_asof_rolling` / `league_asof_rolling`: the mean over each team's most
recent 5 STRICTLY PRIOR completed games, resolved on (team, season, week). Previously each
of these was attached by merging a rolling table on `game_id`, which silently produced NaN
for any game that had not been played — the whole live slate — and a downstream blanket
`fillna(0)` then wrote 0.0 into 10 of the 14 features. On a historical row the as-of lookup
is algebraically the same `shift(1).rolling(N, min_periods=1).mean()` as before; that is
proved by execution in `betting/test_totals_live.py` against the vendored pre-fix builder.

**Imputation.** `impute_missing=True` (default) keeps the legacy slate-mean / zero fills so
the training path and every historical value are unchanged. `impute_missing=False` (the live
path) writes nothing: an unresolved feature stays NaN and `totals_live_preflight` aborts.
Zero is semantically false for a scoring or pace feature and is never substituted.


In [ ]:
def build_totals_features(g, sched, pbp_full, weather_path=None, impute_missing=True):
    """Attach the 14 totals-specific features to `g`.

    Parameters
    ----------
    g : DataFrame
        Games to build features FOR. Needs game_id / season / week / home_team / away_team /
        spread_line / total_line / div_game. Rows may be unplayed.
    sched : DataFrame
        The schedule. **Pass the COMPLETE schedule, including the target week** — it supplies
        the raw `roof` string and the game metadata for the rows being predicted, and it is
        the source of the scoring history. History is derived internally as the rows with
        both scores present, so passing a completed-games-only frame (as the training path
        does) is still valid; it just cannot describe a future game.
    pbp_full : DataFrame
        Unfiltered play-by-play (pace counts every play, not just run/pass).
    weather_path : path or None
        Retained weather CSV (game_id, temp_f, wind_mph). Missing file / missing row is
        handled by the documented fallback below, and the origin of every value is recorded
        in `temp_f_source` / `wind_mph_source`.
    impute_missing : bool, default True
        True  = LEGACY behaviour, kept so the training path and every historical value are
                bit-identical to the pre-2026-08-03 builder: unresolved rolling features are
                filled with the slate mean, weather with the slate mean, div_game with 0.
        False = LIVE behaviour: nothing is silently substituted. An unresolved feature stays
                NaN and `totals_live_preflight` decides whether that is fatal. Zero is
                semantically false for every scoring/pace feature, so it is never written.

    As-of semantics (2026-08-03 fix)
    --------------------------------
    Team scoring, pace and the league environment are STRICTLY-PRIOR as-of lookups keyed on
    (team, season, week) via `totals_asof`. They no longer require the target game's own
    `game_id` to exist in a history table — which is why a game that has not kicked off now
    gets real features instead of NaN-then-0.0. On a historical row the as-of lookup is
    algebraically identical to the old `shift(1).rolling(n, min_periods=1).mean()`; the
    equality is PROVEN by execution in `betting/test_totals_live.py` against the vendored
    pre-fix builder, not asserted here.
    """
    assert 'roof' in sched.columns, \
        "build_totals_features: sched is missing the 'roof' column — is_dome would silently be 0"
    for _c in ['game_id', 'season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']:
        assert _c in sched.columns, f'build_totals_features: sched is missing {_c!r}'
    for _c in ['game_id', 'season', 'week', 'home_team', 'away_team']:
        assert _c in g.columns, f'build_totals_features: g is missing {_c!r}'
    # (season, week) is the as-of key. A null key cannot be resolved, so refuse it outright
    # rather than producing a quietly wrong lookup.
    assert g['season'].notna().all() and g['week'].notna().all(), \
        'build_totals_features: g has null season/week — the as-of key must be complete'

    # ── scores + raw roof string ──────────────────────────────────────────────
    # `g` may already carry home_score/away_score (predict_totals path, where
    # game_rows comes from a full_schedule slice) or may not (totals_model
    # path, where g comes from mc cells which drop the scores). Only pull
    # the score columns from sched when g doesn't already have them — pulling
    # both would create _x/_y suffix collisions.
    _aux_cols = ['game_id', 'roof']
    if 'home_score' not in g.columns:
        _aux_cols += ['home_score', 'away_score']
    aux = sched[_aux_cols].rename(columns={'roof': 'roof_str'})
    g = g.merge(aux, on='game_id', how='left').reset_index(drop=True)
    g['total_points'] = g['home_score'] + g['away_score']

    # ── implied team totals ───────────────────────────────────────────────────
    g['home_implied_pts'] = (g['total_line'] + g['spread_line']) / 2.0
    g['away_implied_pts'] = (g['total_line'] - g['spread_line']) / 2.0

    # ── dome flag (raw string) ────────────────────────────────────────────────
    # LEGACY: an unknown roof was read as 'outdoors'. LIVE: an unknown roof is NaN, because
    # guessing it is precisely how a silently-wrong is_dome shipped before.
    if impute_missing:
        g['is_dome'] = g['roof_str'].fillna('outdoors').isin(['dome', 'closed']).astype(int)
    else:
        g['is_dome'] = np.where(g['roof_str'].notna(),
                                g['roof_str'].isin(['dome', 'closed']).astype(float),
                                np.nan)

    # ── weather: explicit provenance + a documented fallback ─────────────────
    # Resolution order, recorded per row in temp_f_source / wind_mph_source:
    #   1. 'weather_file'     — a retained row in the weather CSV supplied the value.
    #   2. 'dome_neutralized' — dome/closed roof; the value is FORCED to 70F / 0mph and
    #                           overrides any file value. This neutralisation is deliberate
    #                           and is preserved exactly as before.
    #   3. 'default_outdoor'  — nothing supplied a value: the documented outdoor
    #                           league-average constants (60F / 8mph) are used.
    #   4. 'slate_mean'       — LEGACY imputation only (impute_missing=True): the mean of
    #                           this call's own frame. Never used on the live path; on a
    #                           slate of unplayed games that mean is itself undefined, which
    #                           is how all-NaN weather used to slip through.
    _have_wx = weather_path is not None and Path(weather_path).exists()
    if _have_wx:
        wx = pd.read_csv(weather_path)[['game_id', 'temp_f', 'wind_mph']]
        g = g.merge(wx, on='game_id', how='left')
    else:
        g['temp_f'] = np.nan
        g['wind_mph'] = np.nan
    g['temp_f_source'] = np.where(g['temp_f'].notna(), 'weather_file', None)
    g['wind_mph_source'] = np.where(g['wind_mph'].notna(), 'weather_file', None)

    if not _have_wx:
        # No source at all: go straight to the documented outdoor constants, which is what
        # the legacy code did inline (it wrote 60.0 / 8.0 before the dome override).
        g['temp_f'] = g['temp_f'].fillna(TEMP_FALLBACK_F)
        g['wind_mph'] = g['wind_mph'].fillna(WIND_FALLBACK_MPH)
        g['temp_f_source'] = 'default_outdoor'
        g['wind_mph_source'] = 'default_outdoor'

    dome_mask = g['is_dome'] == 1
    g.loc[dome_mask, 'temp_f'] = DOME_TEMP_F
    g.loc[dome_mask, 'wind_mph'] = DOME_WIND_MPH
    g.loc[dome_mask, 'temp_f_source'] = 'dome_neutralized'
    g.loc[dome_mask, 'wind_mph_source'] = 'dome_neutralized'

    if impute_missing:
        _t_missing, _w_missing = g['temp_f'].isna(), g['wind_mph'].isna()
        g['temp_f'] = g['temp_f'].fillna(g['temp_f'].mean())
        g['wind_mph'] = g['wind_mph'].fillna(g['wind_mph'].mean())
        g.loc[_t_missing, 'temp_f_source'] = 'slate_mean'
        g.loc[_w_missing, 'wind_mph_source'] = 'slate_mean'
    else:
        _t_missing, _w_missing = g['temp_f'].isna(), g['wind_mph'].isna()
        g['temp_f'] = g['temp_f'].fillna(TEMP_FALLBACK_F)
        g['wind_mph'] = g['wind_mph'].fillna(WIND_FALLBACK_MPH)
        g.loc[_t_missing, 'temp_f_source'] = 'default_outdoor'
        g.loc[_w_missing, 'wind_mph_source'] = 'default_outdoor'

    # ── history: only games whose scores are known can inform anything ───────
    hist_games = sched[sched['home_score'].notna() & sched['away_score'].notna()]

    # ── rolling pts scored / allowed per team (5-game, strictly prior) ───────
    _h = hist_games[['season', 'week', 'home_team', 'home_score', 'away_score']].rename(
        columns={'home_team': 'team', 'home_score': 'pts_scored', 'away_score': 'pts_allowed'})
    _a = hist_games[['season', 'week', 'away_team', 'away_score', 'home_score']].rename(
        columns={'away_team': 'team', 'away_score': 'pts_scored', 'home_score': 'pts_allowed'})
    team_hist = pd.concat([_h, _a], ignore_index=True)
    tgt_home = g[['home_team', 'season', 'week']].rename(columns={'home_team': 'team'})
    tgt_away = g[['away_team', 'season', 'week']].rename(columns={'away_team': 'team'})
    g['home_pts_scored_5g'] = team_asof_rolling(team_hist, tgt_home, 'pts_scored', 5)
    g['home_pts_allowed_5g'] = team_asof_rolling(team_hist, tgt_home, 'pts_allowed', 5)
    g['away_pts_scored_5g'] = team_asof_rolling(team_hist, tgt_away, 'pts_scored', 5)
    g['away_pts_allowed_5g'] = team_asof_rolling(team_hist, tgt_away, 'pts_allowed', 5)
    g['combined_pts_5g'] = (
        g['home_pts_scored_5g'] + g['home_pts_allowed_5g'] +
        g['away_pts_scored_5g'] + g['away_pts_allowed_5g']) / 4.0
    if impute_missing:
        for c in ['home_pts_scored_5g', 'home_pts_allowed_5g',
                  'away_pts_scored_5g', 'away_pts_allowed_5g', 'combined_pts_5g']:
            g[c] = g[c].fillna(g[c].mean())

    # ── league scoring environment (rolling 4-week avg, strictly prior) ──────
    lg_hist = hist_games[['season', 'week']].copy()
    lg_hist['game_total'] = hist_games['home_score'] + hist_games['away_score']
    g['league_avg_total_4wk'] = league_asof_rolling(
        lg_hist, g[['season', 'week']], 'game_total', window=4)
    if impute_missing:
        g['league_avg_total_4wk'] = g['league_avg_total_4wk'].fillna(
            g['league_avg_total_4wk'].mean())

    # ── pace: rolling plays per game (5-game, both teams averaged) ───────────
    plays = (pbp_full[pbp_full['posteam'].notna()]
             .groupby(['game_id', 'posteam']).size().reset_index(name='plays')
             .rename(columns={'posteam': 'team'}))
    pace_hist = plays.merge(sched[['game_id', 'season', 'week']], on='game_id', how='left')
    pace_hist = pace_hist[pace_hist['season'].notna() & pace_hist['week'].notna()]
    g['home_pace_5g'] = team_asof_rolling(pace_hist, tgt_home, 'plays', 5)
    g['away_pace_5g'] = team_asof_rolling(pace_hist, tgt_away, 'plays', 5)
    g['pace_5g'] = (g['home_pace_5g'] + g['away_pace_5g']) / 2.0
    if impute_missing:
        g['pace_5g'] = g['pace_5g'].fillna(g['pace_5g'].mean())

    # ── div_game (already in g from mc pipeline) ────────────────────────────
    if impute_missing:
        g['div_game'] = g['div_game'].fillna(0).astype(int)
    else:
        # A missing div_game is missing, not 0 — the preflight fails closed on it.
        g['div_game'] = pd.to_numeric(g['div_game'], errors='coerce')

    g = g.drop(columns=['roof_str', 'home_pace_5g', 'away_pace_5g'], errors='ignore')
    return g


## Tests — `build_totals_features`

In [ ]:
if RUN_TESTS:
    _rng = np.random.default_rng(42)
    _n = 20
    _ids = [f'2020_{w}_KC_BUF' for w in range(1, _n + 1)]
    _sched_t = pd.DataFrame({
        'game_id': _ids, 'season': 2020, 'week': range(1, _n + 1),
        'home_team': 'KC', 'away_team': 'BUF',
        'home_score': _rng.integers(14, 40, _n).astype(float),
        'away_score': _rng.integers(14, 40, _n).astype(float),
        'roof': 'outdoors', 'div_game': 0,
    })
    _g_t = pd.DataFrame({
        'game_id': _ids, 'season': 2020, 'week': range(1, _n + 1),
        'home_team': 'KC', 'away_team': 'BUF',
        'spread_line': _rng.uniform(-7, 7, _n),
        'total_line':  _rng.uniform(40, 55, _n),
        'roof': _rng.integers(0, 4, _n),  # ordinal-encoded (as mc does it)
        'div_game': 0,
    })
    _pbp_t = pd.DataFrame({
        'game_id': np.repeat(_ids, 60),
        'posteam': np.tile(['KC', 'BUF'], _n * 30),
    })
    _g_out = build_totals_features(_g_t.copy(), _sched_t, _pbp_t, weather_path=None)
    for _col in TOTALS_FEATURE_COLS:
        assert _col in _g_out.columns, f'Missing: {_col}'
        assert _g_out[_col].notna().all(), f'NaN in {_col}'
    _diff = (_g_out['home_implied_pts'] + _g_out['away_implied_pts'] - _g_out['total_line']).abs().max()
    assert _diff < 1e-6, f'Implied total algebra error: {_diff}'
    assert (_g_out['is_dome'] == 0).all(), 'Outdoor game incorrectly flagged as dome'
    # Test dome detection
    _sched_dome = _sched_t.copy(); _sched_dome['roof'] = 'dome'
    _g_dome = build_totals_features(_g_t.copy(), _sched_dome, _pbp_t, weather_path=None)
    assert (_g_dome['is_dome'] == 1).all(), 'Dome game not detected'
    assert (_g_dome['wind_mph'] == 0).all(), 'Dome game should have wind=0'
    assert (_g_dome['temp_f'] == 70).all(), 'Dome game should have temp=70'
    print(f'build_totals_features tests OK | {len(TOTALS_FEATURE_COLS)} features, dome detection OK, algebra OK')


## Cleanup

In [ ]:
if not RUN_TESTS:
    for _tmp in ['_rng', '_n', '_ids', '_sched_t', '_g_t', '_pbp_t', '_g_out',
                 '_col', '_diff', '_sched_dome', '_g_dome']:
        globals().pop(_tmp, None)
